In [20]:
import pandas as pd 
import numpy as np 
from sklearn.linear_model import Ridge,Lasso,LinearRegression
from sklearn.model_selection import cross_validate,ShuffleSplit
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from sklearn.pipeline import Pipeline
import optuna
from optuna import Trial

In [4]:
df = pd.read_fwf(
    "./data/auto-mpg.data",
    names = ["mpg","cylinders","displacement","horsepower","weight","acceleration","model year","origin","car name"]
);df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
0,18.0,8,307.0,130.0,3504.0,12.0,70,1,"""chevrolet chevelle malibu"""
1,15.0,8,350.0,165.0,3693.0,11.5,70,1,"""buick skylark 320"""
2,18.0,8,318.0,150.0,3436.0,11.0,70,1,"""plymouth satellite"""
3,16.0,8,304.0,150.0,3433.0,12.0,70,1,"""amc rebel sst"""
4,17.0,8,302.0,140.0,3449.0,10.5,70,1,"""ford torino"""


In [5]:
cleaned_df = df.copy()
cleaned_df["horsepower"] = cleaned_df["horsepower"].replace("?",np.nan)
cleaned_df["horsepower"] = pd.to_numeric(cleaned_df["horsepower"],errors="coerce")
cleaned_df["car name"] = cleaned_df["car name"].str.replace("\"","")
cleaned_df.dropna(inplace=True)
cleaned_df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
0,18.0,8,307.0,130.0,3504.0,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693.0,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150.0,3436.0,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150.0,3433.0,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140.0,3449.0,10.5,70,1,ford torino


In [9]:
selected_features:list = ['weight','cylinders','mpg','displacement','acceleration']


In [91]:
MODEL_DICT = {
    "Ridge":Ridge,
    "Lasso" : Lasso,
    "Ols": LinearRegression
}
MODEL_SCALERS = {
    'Min_max':MinMaxScaler(),
    'StandardScaler':StandardScaler(),
    'No_scaling':None
}


In [92]:
X = cleaned_df[selected_features].copy()
y= cleaned_df["horsepower"].copy()
cv_spliter = ShuffleSplit(n_splits=5,test_size=0.2,random_state=42)


In [93]:
def model_selection_objective(trial:Trial):
    model_type = trial.suggest_categorical("model_type",['Ridge','Ols','Lasso'])
    model = MODEL_DICT[model_type]
    scaler_type = trial.suggest_categorical('s',['Min_max','StandardScaler','No_scaling'])
    scaler=MODEL_SCALERS[scaler_type]
    if model_type in ['Ridge','Lasso'] :
        alpha = trial.suggest_float("alpha",0.5,3,step=0.5)
        tol  = trial.suggest_float('tol',1e-4,1e-3,step=1e-4)
        model_instance = model(alpha=alpha,tol=tol,random_state=42)
    else:
        model_instance = model()
    settings = []
    if scaler!=None:
        settings.append(('scaler',scaler))
    settings.append(('model',model_instance))
    model_pipline = Pipeline(settings)
    cv_results = cross_validate(model_pipline,X,y,cv=cv_spliter)
    mean_r2_score = cv_results['test_score'].mean()
    return mean_r2_score


In [94]:
study = optuna.create_study(
    study_name='auto_mpg_model_selection',
    direction='maximize',
    storage='sqlite:///optuna_studies.db',
    load_if_exists=True
)
study.optimize(model_selection_objective,n_trials=100,n_jobs=12,show_progress_bar=True)

[I 2025-09-19 20:11:25,296] A new study created in RDB with name: auto_mpg_model_selection


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2025-09-19 20:11:25,692] Trial 5 finished with value: 0.8906435999305149 and parameters: {'model_type': 'Ridge', 's': 'Min_max', 'alpha': 1.5, 'tol': 0.0004}. Best is trial 5 with value: 0.8906435999305149.
[I 2025-09-19 20:11:25,879] Trial 7 finished with value: 0.8953674695550781 and parameters: {'model_type': 'Ols', 's': 'Min_max'}. Best is trial 7 with value: 0.8953674695550781.
[I 2025-09-19 20:11:25,893] Trial 6 finished with value: 0.8954536669246753 and parameters: {'model_type': 'Ridge', 's': 'StandardScaler', 'alpha': 0.5, 'tol': 0.0009000000000000001}. Best is trial 6 with value: 0.8954536669246753.
[I 2025-09-19 20:11:25,988] Trial 0 finished with value: 0.8953674695550781 and parameters: {'model_type': 'Ols', 's': 'Min_max'}. Best is trial 6 with value: 0.8954536669246753.
[I 2025-09-19 20:11:26,016] Trial 2 finished with value: 0.8906435999305149 and parameters: {'model_type': 'Ridge', 's': 'Min_max', 'alpha': 1.5, 'tol': 0.001}. Best is trial 6 with value: 0.895453666

In [95]:
study.best_params

{'model_type': 'Ridge',
 's': 'StandardScaler',
 'alpha': 3.0,
 'tol': 0.0009000000000000001}

In [96]:
study.best_value

0.8956529576574928

In [97]:
optuna.visualization.plot_slice(study)